In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# 自动向上查找项目根目录 (含 .gitignore 的文件夹)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# 切换 cwd 到项目根, 使所有相对路径 (Stage1_Exploration/, Refined_Results_v4/ 等) 保持有效
os.chdir(PROJECT_ROOT)
# 让 notebooks 能 `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 0. 数据集构建脚本 (Dataset Construction)
# ==========================================
# 功能: 将原始的 CFD 模拟结果 (summary 表) 和工况参数 (cases 表) 
#       合并清洗为机器学习可用的标准数据集 (train_dataset_ready.csv)。
# 输入: 
#   - cases.csv: 每个 Case 的入口条件 (V_in, C_in, Area)
#   - summary_0_499.csv: 每个 Case 的沿程浓度分布数据
# 输出:
#   - train_dataset_ready.csv: 融合后的长表格式数据

# 1. 读取原始数据
print("正在读取原始数据文件...")
try:
    df_cases = pd.read_csv('data/cases.csv')
    df_summary = pd.read_csv('data/summary_0_499.csv')
except FileNotFoundError as e:
    print(f"错误: 找不到文件 {e.filename}。请确保 cases.csv 和 summary_0_499.csv 在当前目录下。")
    exit()

# ==========================================
# 2. 数据重构 (Reshaping: Wide to Long)
# ==========================================
# 原始 summary 表是宽表格式 (每一列代表一个距离点)，不适合模型训练。
# 使用 melt 函数将其转换为长表格式 (每一行代表一个样本点)。

print("正在执行数据透视 (Melting)...")
df_long = df_summary.melt(id_vars=['Case'], 
                          var_name='Distance', 
                          value_name='C_out')

# 3. 类型转换
# Distance 列名默认为字符串，需转换为数值类型以便后续计算
df_long['Distance'] = df_long['Distance'].astype(float)

# ==========================================
# 4. 特征融合 (Feature Merging)
# ==========================================
# 将工况参数 (V_in, C_in, Area) 根据 Case ID 关联到每一个样本点上。
# 这样每个样本点都包含了完整的输入特征 (Input) 和输出目标 (Output)。

print("正在合并工况参数...")
df_merged = pd.merge(df_long, df_cases, on='Case', how='left')

# ==========================================
# 5. 数据清洗与导出 (Cleaning & Export)
# ==========================================
feature_cols = ['V_in', 'C_in', 'Area', 'Distance']
target_col = ['C_out']

# 检查数据完整性
print(f"原始合并数据形状: {df_merged.shape}")
df_merged = df_merged.dropna() 
print(f"清洗后数据形状 (去除空值): {df_merged.shape}")

# 预览数据
print("\n数据预览 (前5行):")
print(df_merged[feature_cols + target_col].head())

# 保存最终数据集
output_file = 'data/train_dataset_ready.csv'
df_merged.to_csv(output_file, index=False)
print(f"\n数据集已成功构建并保存至: {output_file}")

原始数据形状: (55500, 6)
清洗后数据形状: (55500, 6)

数据预览:
       V_in          C_in       Area  Distance         C_out
0  0.446820  3.243668e-06  45.095477    1100.0  9.790578e-07
1  1.866461  1.957847e-07  62.291468    1100.0  1.106667e-07
2  1.017970  1.665882e-07  63.440067    1100.0  7.399183e-08
3  1.973465  5.148324e-07  52.935569    1100.0  2.692419e-07
4  0.913438  8.256297e-07  25.503156    1100.0  1.898755e-07
